In [1]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from langchain_core.messages import BaseMessage,HumanMessage,AIMessage
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

In [2]:
load_dotenv()

True

In [3]:
llm=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [4]:
class chatState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]

In [5]:
def chat(state:chatState):
    response=llm.invoke(state['messages'])
    return {
        'messages':[response]
    }

In [6]:
graph=StateGraph(chatState)

graph.add_node("chat",chat)

graph.add_edge(START,"chat")
graph.add_edge("chat",END)

checkpointer=InMemorySaver()
workflow=graph.compile(checkpointer=checkpointer)




In [7]:
config={
    "configurable":{
        "thread_id":"thread_1"
    }
}
print("Chatbot is ready! Type 'exit' to stop.\n")
while(True):
    
    userInput=input("You : ")
    if(userInput.lower()=="exit"):
        print("chatBot : GoodBye")
        break
    print(f"\nYou: {userInput}")
    print("Bot: ", end="", flush=True)
    
    for message_chunks,metadata in workflow.stream(
        {
            "messages":[
                HumanMessage(content=userInput)
            ]
        },
        config=config,
        stream_mode="messages"
    ):
        if metadata.get("langgraph_node")=="chat":
            content = message_chunks.content

            if isinstance(content, str):
                print(content, end="", flush=True)

            elif isinstance(content, list):
                for block in content:
                    if block.get("type") == "text":
                        print(block.get("text", ""), end="", flush=True)

    print()

Chatbot is ready! Type 'exit' to stop.


You: hii
Bot: Hello! How can I help you today?

You: what is my name
Bot: I don't know your name because you haven't told me yet! What should I call you?

You: my name is sai
Bot: Nice to meet you, Sai! How can I help you today?
chatBot : GoodBye
